In [ ]:
%pip -q install google-cloud-bigquery db-dtypes pandas numpy scipy statsmodels


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel
from statsmodels.stats.proportion import proportions_ztest
from IPython.display import display


In [ ]:
metrics = {
    "add_payment_info / session": "add_payment_info",
    "add_shipping_info / session": "add_shipping_info",
    "begin_checkout / session": "begin_checkout",
    "new_accounts / session": "new_accounts",
}

metric_columns = list(metrics.values())
alpha = 0.05
SOURCE_DATASET = "data-analytics-mate.DA"


In [ ]:
from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()
client = bigquery.Client(project="data-analytics-mate")



In [ ]:
query = f"""
WITH session_info AS (
  Select
    s.date,
    s.ga_session_id,
    sp.country,
    sp.device,
    sp.continent,
    sp.channel,
    ab.test,
    ab.test_group
  from `DA.ab_test` ab
  join `DA.session` s
    on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` sp
    on sp.ga_session_id = ab.ga_session_id
),


session_with_orders as (
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct o.ga_session_id) session_with_orders_cnt
from `DA.order` o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
events as (
SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name,
    count(ep.ga_session_id) event_cnt
from `DA.event_params` ep
join session_info
on ep.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name
),
 session as(
SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct session_info.ga_session_id) session_cnt
FROM session_info
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
 ),
 new_accounts as (
 SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    count(distinct acs.ga_session_id) new_account_cnt
FROM `DA.account_session` acs
JOIN session_info
on acs.ga_session_id = session_info.ga_session_id
group by
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
 )
  SELECT
    session_with_orders.date,
    session_with_orders.country,
    session_with_orders.device,
    session_with_orders.continent,
    session_with_orders.channel,
    session_with_orders.test,
    session_with_orders.test_group,
    'session with orders' as event_name,
    session_with_orders.session_with_orders_cnt as value
    from session_with_orders


    UNION ALL


SELECT
    events.date,
    events.country,
    events.device,
    events.continent,
    events.channel,
    events.test,
    events.test_group,
    events.event_name,
    events.event_cnt as value
    from events

UNION ALL

   SELECT
    session.date,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' as event_name,
    session.session_cnt as value
    from session

UNION ALL

   SELECT
    new_accounts.date,
    new_accounts.country,
    new_accounts.device,
    new_accounts.continent,
    new_accounts.channel,
    new_accounts.test,
    new_accounts.test_group,
    'new accounts' as event_name,
    new_accounts.new_account_cnt as value
    from new_accounts
"""
df = client.query(query).to_dataframe(create_bqstorage_client=False)
df.head()


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-06,(not set),mobile,Asia,Paid Search,2,2,new accounts,1
1,2020-11-13,(not set),mobile,Europe,Undefined,2,2,new accounts,1
2,2020-11-17,(not set),mobile,Asia,Organic Search,2,2,new accounts,1
3,2020-11-22,(not set),desktop,(not set),Undefined,2,2,new accounts,1
4,2020-11-28,(not set),mobile,Asia,Organic Search,2,2,new accounts,1


## 6. Control and variant

Group 1 is control A; group 2 is variant B, as confirmed in the original dashboard. The comparison is B versus A. A positive test statistic indicates a higher metric value in B.


In [ ]:
control_group = 1
variant_group = 2

## 7. Total counts for each test and group



In [ ]:
summary = df.pivot_table(
    index=["test", "test_group"],
    columns="event_name",
    values="value",
    aggfunc="sum",
    fill_value=0
)

summary = summary.rename(columns={"new accounts": "new_accounts"})

display(summary[["session"] + metric_columns])

event_name       session  add_payment_info  add_shipping_info  begin_checkout  \
test test_group                                                                 
1    1             45362              1988               3034            3784   
     2             45193              2229               3221            4021   
2    1             50637              2344               3480            4262   
     2             50244              2409               3510            4313   
3    1             70047              3623               5298            9532   
     2             70439              3697               5188            9264   
4    1            105079              3731               5128           12555   
     2            105141              3601               4956           12267   

event_name       new_accounts  
test test_group                
1    1                   3823  
     2                   3681  
2    1                   4165  
     2                   4184  
3    1                   5856  
     2                   5822  
4    1                   8984  
     2                   8687

## 8. Metric values per session


In [ ]:
metric_rates = summary[metric_columns].copy()

for metric in metric_columns:
    metric_rates[metric] = summary[metric] / summary["session"]

display((metric_rates * 100).round(2))


event_name       add_payment_info  add_shipping_info  begin_checkout  \
test test_group                                                        
1    1                       4.38               6.69            8.34   
     2                       4.93               7.13             8.9   
2    1                       4.63               6.87            8.42   
     2                       4.79               6.99            8.58   
3    1                       5.17               7.56           13.61   
     2                       5.25               7.37           13.15   
4    1                       3.55               4.88           11.95   
     2                       3.42               4.71           11.67   

event_name       new_accounts  
test test_group                
1    1                   8.43  
     2                   8.15  
2    1                   8.23  
     2                   8.33  
3    1                   8.36  
     2                   8.27  
4    1                   8.55  
     2                   8.26

---PANDAS---
```
metric_rates = summary[metric_columns].div(
    summary["session"], axis="index"
    )
display((metric_rates * 100).round(2))
```

## 9. Z-test and paired t-test



In [ ]:
daily = df.pivot_table(
    index=["test", "test_group", "date"],
    columns="event_name", values="value",
    aggfunc="sum", fill_value=0
)
daily = daily.rename(columns={"new accounts": "new_accounts"})

daily_rates = daily[metric_columns].copy()
for metric in metric_columns:
    daily_rates[metric] = daily[metric] / daily["session"]

display(daily_rates.head())


event_name                  add_payment_info  add_shipping_info  \
test test_group date                                              
1    1          2020-11-01          0.026336            0.03988   
                2020-11-02           0.02718           0.051529   
                2020-11-03          0.026521           0.039002   
                2020-11-04          0.035316           0.045539   
                2020-11-05          0.029085           0.063458   

event_name                  begin_checkout  new_accounts  
test test_group date                                      
1    1          2020-11-01         0.03988      0.087284  
                2020-11-02        0.051529       0.08607  
                2020-11-03        0.039002      0.088534  
                2020-11-04        0.045539      0.083178  
                2020-11-05        0.063458      0.085669

In [ ]:
results = []

for test in sorted(df["test"].unique()):
    a = summary.loc[test].loc[control_group]
    b = summary.loc[test].loc[variant_group]
    a_daily = daily_rates.loc[test].loc[control_group]
    b_daily = daily_rates.loc[test].loc[variant_group]

    for metric in metric_columns:
        if metric == "new_accounts":
            counts = [b[metric], a[metric]]
            sessions = [b["session"], a["session"]]
            statistic, p = proportions_ztest(counts, sessions)
            method = "Two-proportion z-test"
            mean_daily_a, mean_daily_b, paired_days = np.nan, np.nan, np.nan
        else:
            pairs = pd.DataFrame({"A": a_daily[metric], "B": b_daily[metric]}).dropna()
            statistic, p = ttest_rel(pairs["B"], pairs["A"])
            method = "Paired t-test (daily rates)"
            mean_daily_a = pairs["A"].mean()
            mean_daily_b = pairs["B"].mean()
            paired_days = len(pairs)

        results.append([
            test, metric, method, statistic, p,
            mean_daily_a, mean_daily_b, paired_days
        ])

results = pd.DataFrame(results, columns=[
    "test", "metric", "test_method", "test_statistic", "p_value",
    "mean_daily_a", "mean_daily_b", "paired_days"
])

display(results)


,test,metric,test_method,test_statistic,p_value,mean_daily_a,mean_daily_b,paired_days
0,1,add_payment_info,Paired t-test (daily rates),1.847745,0.076505,0.043234,0.048819,26.0
1,1,add_shipping_info,Paired t-test (daily rates),1.372147,0.182205,0.066925,0.071271,26.0
2,1,begin_checkout,Paired t-test (daily rates),1.490516,0.148599,0.084518,0.090682,26.0
3,1,new_accounts,Two-proportion z-test,-1.542883,0.122859,NaN,NaN,NaN
4,2,add_payment_info,Paired t-test (daily rates),1.152425,0.258891,0.045140,0.048146,29.0
5,2,add_shipping_info,Paired t-test (daily rates),0.692858,0.494106,0.068297,0.070263,29.0
6,2,begin_checkout,Paired t-test (daily rates),0.775623,0.444474,0.084579,0.087934,29.0
7,2,new_accounts,Two-proportion z-test,0.588793,0.556000,NaN,NaN,NaN
8,3,add_payment_info,Paired t-test (daily rates),0.531352,0.598962,0.049769,0.051090,32.0
9,3,add_shipping_info,Paired t-test (daily rates),-0.828710,0.413604,0.074555,0.072399,32.0


## 10. Bonferroni adjustment


In [ ]:
results["p_value_bonferroni"] = (
    results["p_value"] * len(metric_columns)
).clip(upper=1)

results["significant"] = (results["p_value_bonferroni"] < alpha).astype("boolean")
results.loc[results["p_value_bonferroni"].isna(), "significant"] = pd.NA

display(results)


,test,metric,test_method,test_statistic,p_value,mean_daily_a,mean_daily_b,paired_days,p_value_bonferroni,significant
0,1,add_payment_info,Paired t-test (daily rates),1.847745,0.076505,0.043234,0.048819,26.0,0.306021,False
1,1,add_shipping_info,Paired t-test (daily rates),1.372147,0.182205,0.066925,0.071271,26.0,0.728821,False
2,1,begin_checkout,Paired t-test (daily rates),1.490516,0.148599,0.084518,0.090682,26.0,0.594395,False
3,1,new_accounts,Two-proportion z-test,-1.542883,0.122859,NaN,NaN,NaN,0.491437,False
4,2,add_payment_info,Paired t-test (daily rates),1.152425,0.258891,0.045140,0.048146,29.0,1.000000,False
5,2,add_shipping_info,Paired t-test (daily rates),0.692858,0.494106,0.068297,0.070263,29.0,1.000000,False
6,2,begin_checkout,Paired t-test (daily rates),0.775623,0.444474,0.084579,0.087934,29.0,1.000000,False
7,2,new_accounts,Two-proportion z-test,0.588793,0.556000,NaN,NaN,NaN,1.000000,False
8,3,add_payment_info,Paired t-test (daily rates),0.531352,0.598962,0.049769,0.051090,32.0,1.000000,False
9,3,add_shipping_info,Paired t-test (daily rates),-0.828710,0.413604,0.074555,0.072399,32.0,1.000000,False


## 11. Pooled values and daily-test estimates



In [ ]:
for i in results.index:
    test = results.loc[i, "test"]
    metric = results.loc[i, "metric"]

    a = metric_rates.loc[test].loc[control_group]
    b = metric_rates.loc[test].loc[variant_group]

    results.loc[i, "rate_a"] = a[metric]
    results.loc[i, "rate_b"] = b[metric]

results["difference_per_100_sessions"] = (
    results["rate_b"] - results["rate_a"]
) * 100

results["daily_difference_per_100_sessions"] = (
    results["mean_daily_b"] - results["mean_daily_a"]
) * 100

display(results)


,test,metric,test_method,test_statistic,p_value,mean_daily_a,mean_daily_b,paired_days,p_value_bonferroni,significant,rate_a,rate_b,difference_per_100_sessions,daily_difference_per_100_sessions
0,1,add_payment_info,Paired t-test (daily rates),1.847745,0.076505,0.043234,0.048819,26.0,0.306021,False,0.043825,0.049322,0.549657,0.558538
1,1,add_shipping_info,Paired t-test (daily rates),1.372147,0.182205,0.066925,0.071271,26.0,0.728821,False,0.066884,0.071272,0.438792,0.434568
2,1,begin_checkout,Paired t-test (daily rates),1.490516,0.148599,0.084518,0.090682,26.0,0.594395,False,0.083418,0.088974,0.555612,0.616398
3,1,new_accounts,Two-proportion z-test,-1.542883,0.122859,NaN,NaN,NaN,0.491437,False,0.084278,0.081451,-0.282692,NaN
4,2,add_payment_info,Paired t-test (daily rates),1.152425,0.258891,0.045140,0.048146,29.0,1.000000,False,0.046290,0.047946,0.165576,0.300653
5,2,add_shipping_info,Paired t-test (daily rates),0.692858,0.494106,0.068297,0.070263,29.0,1.000000,False,0.068724,0.069859,0.113464,0.196648
6,2,begin_checkout,Paired t-test (daily rates),0.775623,0.444474,0.084579,0.087934,29.0,1.000000,False,0.084168,0.085841,0.167339,0.335493
7,2,new_accounts,Two-proportion z-test,0.588793,0.556000,NaN,NaN,NaN,1.000000,False,0.082252,0.083274,0.102152,NaN
8,3,add_payment_info,Paired t-test (daily rates),0.531352,0.598962,0.049769,0.051090,32.0,1.000000,False,0.051722,0.052485,0.076271,0.132100
9,3,add_shipping_info,Paired t-test (daily rates),-0.828710,0.413604,0.074555,0.072399,32.0,1.000000,False,0.075635,0.073652,-0.198255,-0.215593


In [ ]:
results["relative_change"] = (
    results["rate_b"] / results["rate_a"] - 1
)

display(results)

,test,metric,test_method,test_statistic,p_value,mean_daily_a,mean_daily_b,paired_days,p_value_bonferroni,significant,rate_a,rate_b,difference_per_100_sessions,daily_difference_per_100_sessions,relative_change
0,1,add_payment_info,Paired t-test (daily rates),1.847745,0.076505,0.043234,0.048819,26.0,0.306021,False,0.043825,0.049322,0.549657,0.558538,0.125420
1,1,add_shipping_info,Paired t-test (daily rates),1.372147,0.182205,0.066925,0.071271,26.0,0.728821,False,0.066884,0.071272,0.438792,0.434568,0.065605
2,1,begin_checkout,Paired t-test (daily rates),1.490516,0.148599,0.084518,0.090682,26.0,0.594395,False,0.083418,0.088974,0.555612,0.616398,0.066606
3,1,new_accounts,Two-proportion z-test,-1.542883,0.122859,NaN,NaN,NaN,0.491437,False,0.084278,0.081451,-0.282692,NaN,-0.033543
4,2,add_payment_info,Paired t-test (daily rates),1.152425,0.258891,0.045140,0.048146,29.0,1.000000,False,0.046290,0.047946,0.165576,0.300653,0.035769
5,2,add_shipping_info,Paired t-test (daily rates),0.692858,0.494106,0.068297,0.070263,29.0,1.000000,False,0.068724,0.069859,0.113464,0.196648,0.016510
6,2,begin_checkout,Paired t-test (daily rates),0.775623,0.444474,0.084579,0.087934,29.0,1.000000,False,0.084168,0.085841,0.167339,0.335493,0.019882
7,2,new_accounts,Two-proportion z-test,0.588793,0.556000,NaN,NaN,NaN,1.000000,False,0.082252,0.083274,0.102152,NaN,0.012419
8,3,add_payment_info,Paired t-test (daily rates),0.531352,0.598962,0.049769,0.051090,32.0,1.000000,False,0.051722,0.052485,0.076271,0.132100,0.014746
9,3,add_shipping_info,Paired t-test (daily rates),-0.828710,0.413604,0.074555,0.072399,32.0,1.000000,False,0.075635,0.073652,-0.198255,-0.215593,-0.026212


In [ ]:
for i in results.index:
    test = results.loc[i, "test"]
    metric = results.loc[i, "metric"]

    a = summary.loc[test].loc[control_group]
    b = summary.loc[test].loc[variant_group]

    results.loc[i, "sessions_a"] = a["session"]
    results.loc[i, "sessions_b"] = b["session"]
    results.loc[i, "count_a"] = a[metric]
    results.loc[i, "count_b"] = b[metric]

count_columns = ["sessions_a", "sessions_b", "count_a", "count_b"]
results[count_columns] = results[count_columns].astype(int)

In [ ]:
from google.colab import files

results.to_csv("ab_test_results.csv", index=False)
files.download("ab_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

https://public.tableau.com/app/profile/maxim.voronenko2142/viz/PROJECT02_17947675861610/ExperimentOverview?publish=yes

###```Не надаю доступу до редагування, оскільки в дашборді Tableau є пряме посилання на код у Google Colab.```